# 1- Importing Libraries

In [1]:
! pip install pandas
! pip install numpy
! pip install scikit-learn
! pip install nltk
! pip install seaborn
! pip install imbalanced-learn
! pip install langchain
! pip install langchain-huggingface
! pip install sentence-transformers
! pip install xgboost
! pip install python-dotenv

In [1]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import nltk
import seaborn as sns
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from nltk.tokenize import sent_tokenize,word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from imblearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,f1_score,classification_report,confusion_matrix
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer
nltk.download("stopwords")
nltk.download("punkt_tab")
lemmatizer = WordNetLemmatizer()
nltk.download('wordnet')
load_dotenv()

/Users/prabhsandhu/Downloads/ML Final Project/newenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/prabhsandhu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/prabhsandhu/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/prabhsandhu/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


False

# 2- Loading Dataset

In [10]:
df = pd.read_csv("Reviews.csv")
df = df.sample(50000)

# 3- Understanding Dataset

In [8]:
df.columns

Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text'],
      dtype='str')

In [31]:
df.shape

(200000, 10)

In [ ]:
df.nunique()
df["Score"].unique()

array([5, 1, 4, 2, 3])

In [ ]:
df['HelpfulnessNumerator']

0         1
1         0
2         1
3         3
4         0
         ..
568449    0
568450    0
568451    2
568452    1
568453    0
Name: HelpfulnessNumerator, Length: 568454, dtype: int64

# 4- Preprocessing the Dataset

In [4]:
df.isnull().sum()

Id                        0
ProductId                 0
UserId                    0
ProfileName               4
HelpfulnessNumerator      0
HelpfulnessDenominator    0
Score                     0
Time                      0
Summary                   0
Text                      0
dtype: int64

In [11]:
df.dropna(inplace=True)

In [12]:
df.duplicated().sum()

np.int64(0)

In [13]:
df.drop(['Id','UserId','Time','Summary','HelpfulnessNumerator','HelpfulnessDenominator','ProfileName'],axis=1,inplace=True)

In [14]:
df["Sentiment"] = df["Score"].apply(lambda x:"Negative" if x<=2 else ("Neutral" if x==3 else "Positive"))

In [15]:
df.drop("Score",axis=1,inplace=True)

# 5- Text Preprocessing Using NLP

In [16]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))  # load ONCE

text_preprocessed = []

for text in tqdm(df["Text"].astype(str)):  # ensure all rows are strings
    text = text.lower()
    text = re.sub(r'[^A-Za-z0-9]', ' ', text)
    
    words = word_tokenize(text)
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words and len(word) >= 2]
    
    text_preprocessed.append(" ".join(words))

100%|██████████| 49994/49994 [00:22<00:00, 2189.07it/s]


# 6- Training Machine Learning Models Using LLM Embeddings as Technique

In [17]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sentence_transformers import SentenceTransformer


X = text_preprocessed
y = df["Sentiment"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)

model = SentenceTransformer("sentence-transformers/all-distilroberta-v1")

X_embeddings = model.encode(
    X,
    batch_size=16,           
    convert_to_numpy=True,
    show_progress_bar=True
)

X_train, X_test, y_train, y_test = train_test_split(
    X_embeddings,
    y_encoded,
    test_size=0.3,
    random_state=42,
    stratify=y_encoded
)

clf = LogisticRegression(
    max_iter=3000,
    class_weight="balanced"
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Batches: 100%|██████████| 3125/3125 [05:52<00:00,  8.87it/s] 


Accuracy: 0.7113807587172478
              precision    recall  f1-score   support

           0       0.51      0.67      0.58      2153
           1       0.19      0.52      0.28      1117
           2       0.95      0.74      0.83     11729

    accuracy                           0.71     14999
   macro avg       0.55      0.64      0.56     14999
weighted avg       0.83      0.71      0.75     14999



# 7- Testing Model

In [30]:
prompt = df["Text"].iloc[4800]

embedding = model.encode(
    [prompt],
    convert_to_numpy=True
)

pred = clf.predict(embedding)

print("Text:", prompt)
print("Predicted class:", le.inverse_transform(pred))

Text: This was a great deal! they taste fresh and they were intact and shiny and at a great bulk price! I used them in my wedding favor bags for 300 guests and I put white and gold 5 pieced in each bag. They were a hit!
Predicted class: ['Positive']
